# Installing Dependencies

In [ ]:
!pip install -r /content/drive/MyDrive/418-Hackathon/requirements.txt

In [ ]:
# Run this cell FIRST before any imports
!pip install -q "lightly>=1.5.0" "pydantic==2.7.1"

In [ ]:
!pip install terratorch>=1.2.4
!pip install diffusers==0.30.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 58.9 MB/s eta 0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.37.1
    Uninstalling diffusers-0.37.1:
      Successfully uninstalled diffusers-0.37.1


### Patch for Google Colab

In [ ]:
# Patch pydantic v1 compat for Python 3.12
import pydantic.v1.typing as _t
import typing

_original = typing.ForwardRef._evaluate

def _patched_evaluate(self, globalns, localns, *args, **kwargs):
    kwargs.setdefault('recursive_guard', frozenset())
    return _original(self, globalns, localns, *args, **kwargs)

typing.ForwardRef._evaluate = _patched_evaluate

In [ ]:
import subprocess
result = subprocess.run(['python', '-c', 'import pydantic.v1.typing; print(pydantic.v1.typing.__file__)'],
                      capture_output=True, text=True)
print(result.stdout.strip())

/usr/local/lib/python3.12/dist-packages/pydantic/v1/typing.py


In [ ]:
import re

filepath = '/usr/local/lib/python3.12/dist-packages/pydantic/v1/typing.py'

with open(filepath, 'r') as f:
    content = f.read()

# The broken line calls _evaluate with 3 positional args
old = "return cast(Any, type_)._evaluate(globalns, localns, set())"
new = "return cast(Any, type_)._evaluate(globalns, localns, recursive_guard=set())"

if old in content:
    content = content.replace(old, new)
    with open(filepath, 'w') as f:
        f.write(content)
    print("Patched successfully!")
else:
    print("Line not found — already patched or different version")

Patched successfully!


# Model Running

In [ ]:
!python /content/drive/MyDrive/418-Hackathon/sample_input/generate_synthetic.py

Generating synthetic SAR tile …
✅  Saved: /content/drive/MyDrive/418-Hackathon/sample_input/sample_sar.npy
   Shape : (2, 512, 512)  dtype: float32
   Size  : 2.00 MB
   VV range: [0.0000, 1.0000]
   VH range: [0.0000, 0.9454]

Run inference with:
  python infer.py "backend/mIoU=0.78.ckpt" "sample_input/sample_sar.npy"


In [ ]:
!python /content/drive/MyDrive/418-Hackathon/infer.py "/content/drive/MyDrive/418-Hackathon/backend/mIoU=0.78.ckpt" "/content/drive/MyDrive/418-Hackathon/sample_input/sample_sar.npy"

22:55:02 | INFO     | Loading TerraMind checkpoint …
22:55:04 | INFO     | NumExpr defaulting to 2 threads.
22:55:19 | WARNING  | CUDA not available — loading checkpoint on CPU. Inference will be slow (~30 s per tile).
22:55:19 | INFO     | Loading checkpoint from: /content/drive/MyDrive/418-Hackathon/backend/mIoU=0.78.ckpt
22:55:20 | INFO     | HTTP Request: HEAD https://huggingface.co/ibm-esa-geospatial/TerraMind-1.0-tiny/resolve/main/TerraMind_v1_tiny.pt "HTTP/1.1 302 Found"
22:55:20 | INFO     | Loaded weights for VV in position 0 of patch embed
22:55:20 | INFO     | Loaded weights for VH in position 1 of patch embed
22:55:20 | INFO     | TerraMind checkpoint loaded successfully on cpu.
22:55:20 | INFO     | Model ready in 18.1 s on CPU
22:55:20 | INFO     | Loading SAR input: /content/drive/MyDrive/418-Hackathon/sample_input/sample_sar.npy
22:55:20 | INFO     | Loaded .npy SAR array: shape=(2, 512, 512) dtype=float32
22:55:20 | INFO     | Input shape: (2, 512, 512)  dtype: float32